![The Same Function in LangSmith](assets/banner-03.svg)

**LangSmith** is the observability tool from the team behind LangChain. It is a hosted website, so there is nothing to install: you create a free account and get an API key.

We trace the **same weather assistant** again. You will see that the code changes look almost identical to Langfuse. That is the point of this notebook: once you understand traces and spans, every tool feels familiar.

## Step 1: Get a key

1. Sign up at [smith.langchain.com](https://smith.langchain.com)
2. Go to **Settings, API Keys** and create a key
3. Add these lines to the `.env` file, then restart this notebook's kernel:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=weather-assistant
```

`LANGSMITH_PROJECT` is just a folder name for your traces.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv("../.env")

if os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith key found. Traces will go to project:", os.getenv("LANGSMITH_PROJECT", "default"))
else:
    print("No LangSmith key yet. The code below still runs; it just does not send anything.")

No LangSmith key yet. The code below still runs; it just does not send anything.


## Step 2: Add LangSmith to the weather assistant

Compare with notebook 02. Again, only two things change (marked `# NEW`):

| Langfuse | LangSmith | Purpose |
|---|---|---|
| `from langfuse.openai import OpenAI` | `client = wrap_openai(OpenAI())` | Record every AI call |
| `@observe()` | `@traceable` | Turn a function into a span |

In [2]:
import time

from langsmith import traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI

client = wrap_openai(OpenAI())  # NEW: records every AI call

FAKE_WEATHER = {"Paris": "rainy, 14°C", "Tokyo": "sunny, 21°C", "London": "stormy, 11°C", "Dubai": "hot and dry, 35°C"}


@traceable  # NEW: this function becomes a span
def get_weather(city):
    time.sleep(0.5)
    return FAKE_WEATHER[city]


@traceable  # NEW
def ask_ai(question):
    reply = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
    )
    return reply.choices[0].message.content


@traceable  # NEW: the outer function becomes the whole trace
def weather_assistant(city):
    weather = get_weather(city)
    question = f"The weather in {city} is {weather}. Should I carry an umbrella? Answer in one short sentence."
    return ask_ai(question)

## Step 3: Run it

In [3]:
print(weather_assistant("Paris"))
print(weather_assistant("Tokyo"))

Yes, you should carry an umbrella.


No, you don't need to carry an umbrella.


## Step 4: Look at the trace

Open [smith.langchain.com](https://smith.langchain.com), click **Tracing**, then your project `weather-assistant`. Click the newest `weather_assistant` row.

You will see the same tree as in Langfuse:

```text
weather_assistant            1.4 s
├── get_weather              0.5 s
└── ask_ai                   0.9 s
    └── ChatOpenAI           0.9 s   52 tokens   $0.00002
```

## Same ideas, different words

Each tool uses its own vocabulary for the same things. Do not let it confuse you:

| What it is | OpenTelemetry | Langfuse | LangSmith |
|---|---|---|---|
| One request, start to finish | Trace | Trace | Trace |
| One step | Span | Observation | Run |
| An AI model call | Span with `gen_ai` details | Generation | LLM run |
| A folder of traces | Service | Project | Project |
| A conversation of many requests | | Session | Thread |

## Langfuse or LangSmith?

| | Langfuse | LangSmith |
|---|---|---|
| Price and license | Free and open source | Free tier, paid plans |
| Where it runs | Your laptop or server, or their cloud | Their cloud |
| Best when | You want control of your data | You already use LangChain or LangGraph |

Both are good choices. Pick the one your team will actually open every day.

**Next:** [04 Other tools](04_other_tools.ipynb). These two are not the only options.